# Full Pipeline Demo
End-to-end pipeline run with report outputs.

This notebook runs the complete `PipelineOrchestrator` and inspects:
- Execution manifest (stage timings, status)
- Daily, weekly, and monthly reports
- Warehouse table statistics
- DQ score and anomaly summary


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import sqlite3
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 100)

from datetime import datetime, timedelta
from config import BASE_DIR, WAREHOUSE_DB

print(f"Project root : {BASE_DIR}")
print(f"Warehouse DB : {WAREHOUSE_DB}")
print(f"Run started  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
from src.logger import setup_logging
from src.orchestrator import PipelineOrchestrator

setup_logging()

# Use a short date range to keep demo fast
END   = datetime(2025, 3, 1)
START = datetime(2025, 1, 15)

print(f"Running full pipeline: {START.date()} → {END.date()}")
print("(Smaller volumes configured via extractor defaults)\n")

orchestrator = PipelineOrchestrator()
manifest = orchestrator.run_pipeline(START, END)

print()
print(f"Pipeline status : {manifest.get('status')}")
print(f"Run ID          : {manifest.get('run_id')}")
print(f"DQ Score        : {manifest.get('dq_score', 'N/A')}")
print(f"Anomalies found : {manifest.get('anomalies_found', 'N/A')}")


In [ ]:
# ── Pipeline execution manifest ───────────────────────────────────────────
print("=== Pipeline Execution Manifest ===")
print(f"Run ID    : {manifest.get('run_id')}")
print(f"Status    : {manifest.get('status')}")
print(f"Started   : {manifest.get('started_at')}")
print(f"Completed : {manifest.get('completed_at', 'N/A')}")
print()

stages = manifest.get('stages', [])
if stages:
    stages_df = pd.DataFrame(stages)
    display_cols = [c for c in ['stage', 'status', 'duration_seconds',
                                  'rows_processed', 'error'] if c in stages_df.columns]
    print("Stage timings:")
    display(stages_df[display_cols].set_index('stage') if 'stage' in stages_df.columns else stages_df)
else:
    print("No stage detail available in manifest.")
    print("Manifest keys:", list(manifest.keys()))


In [ ]:
# ── Daily report ──────────────────────────────────────────────────────────
from src.reporting import DailyReport

daily_rpt = DailyReport()
try:
    daily_output = daily_rpt.generate(report_date=END)
    if isinstance(daily_output, dict):
        content = daily_output.get('content', daily_output.get('report', str(daily_output)))
    else:
        content = str(daily_output)
    print("=== DAILY REPORT ===")
    print(content[:3000])
    if len(content) > 3000:
        print(f"\n... (truncated – {len(content)} chars total)")
except Exception as e:
    print(f"Daily report error: {e}")
    print("Tip: Run the full pipeline first so the warehouse has data for this date.")


In [ ]:
# ── Weekly report ─────────────────────────────────────────────────────────
from src.reporting import WeeklyReport

weekly_rpt = WeeklyReport()
try:
    weekly_output = weekly_rpt.generate(report_date=END)
    if isinstance(weekly_output, dict):
        content = weekly_output.get('content', weekly_output.get('report', str(weekly_output)))
    else:
        content = str(weekly_output)
    print("=== WEEKLY REPORT ===")
    print(content[:3000])
    if len(content) > 3000:
        print(f"\n... (truncated – {len(content)} chars total)")
except Exception as e:
    print(f"Weekly report error: {e}")


In [ ]:
# ── Warehouse table statistics ─────────────────────────────────────────────
try:
    conn = sqlite3.connect(WAREHOUSE_DB)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    all_tables = [row[0] for row in cursor.fetchall()]

    stats = []
    for tname in all_tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {tname}")
            row_count = cursor.fetchone()[0]
            cursor.execute(f"PRAGMA table_info({tname})")
            col_count = len(cursor.fetchall())
            stats.append({"Table": tname, "Rows": row_count, "Columns": col_count})
        except Exception:
            pass

    conn.close()

    if stats:
        stats_df = pd.DataFrame(stats)
        total_rows = stats_df['Rows'].sum()
        print("=== Warehouse Table Statistics ===")
        display(stats_df.set_index("Table"))
        print(f"\nTotal rows across all tables: {total_rows:,}")
    else:
        print("No tables found – warehouse may be empty.")
except Exception as e:
    print(f"Warehouse stats error: {e}")


In [ ]:
# ── DQ score and anomaly summary ──────────────────────────────────────────
dq_score        = manifest.get('dq_score', 0)
anomalies_found = manifest.get('anomalies_found', 0)
pipeline_status = manifest.get('status', 'UNKNOWN')

print("=== DQ Score & Anomaly Summary ===")
print(f"  Pipeline Status : {pipeline_status}")
print(f"  DQ Score        : {dq_score}/100")
print(f"  Anomalies Found : {anomalies_found}")

# Simple gauge visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Pipeline Health Summary", fontsize=14, fontweight='bold')

# DQ Score gauge (horizontal bar)
color = '#4CAF50' if dq_score >= 90 else ('#FF9800' if dq_score >= 70 else '#F44336')
axes[0].barh(['DQ Score'], [dq_score], color=color, height=0.4)
axes[0].barh(['DQ Score'], [100],      color='#E0E0E0', height=0.4, zorder=0)
axes[0].set_xlim(0, 100)
axes[0].set_title("Data Quality Score")
axes[0].set_xlabel("Score / 100")
axes[0].axvline(x=70, color='orange', linestyle='--', linewidth=1, label='Warning threshold (70)')
axes[0].axvline(x=90, color='green',  linestyle='--', linewidth=1, label='Pass threshold (90)')
axes[0].text(dq_score + 1, 0, f"{dq_score}", va='center', fontweight='bold')
axes[0].legend(fontsize=8)

# Anomaly count bar
anom_color = '#F44336' if anomalies_found > 10 else ('#FF9800' if anomalies_found > 0 else '#4CAF50')
axes[1].bar(['Anomalies'], [anomalies_found], color=anom_color, width=0.3)
axes[1].set_title("Anomalies Detected")
axes[1].set_ylabel("Count")
axes[1].text(0, anomalies_found + 0.3, str(anomalies_found), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(BASE_DIR / "visuals" / "nb06_pipeline_health.png", dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved.")


In [ ]:
# ── Final summary statistics ──────────────────────────────────────────────
print("=" * 60)
print("FULL PIPELINE DEMO – FINAL SUMMARY")
print("=" * 60)
print(f"Run ID          : {manifest.get('run_id', 'N/A')}")
print(f"Pipeline Status : {manifest.get('status', 'N/A')}")
print(f"Date Range      : {START.date()} → {END.date()}")
print(f"DQ Score        : {manifest.get('dq_score', 'N/A')}/100")
print(f"Anomalies       : {manifest.get('anomalies_found', 'N/A')}")
print()

# Stage summary
stages = manifest.get('stages', [])
if stages:
    passed_stages = [s for s in stages if s.get('status') == 'SUCCESS']
    failed_stages = [s for s in stages if s.get('status') == 'FAILED']
    print(f"Stages passed   : {len(passed_stages)}/{len(stages)}")
    if failed_stages:
        print(f"Stages failed   : {[s.get('stage') for s in failed_stages]}")

print()
print("Reports generated:")
reports_dir = BASE_DIR / "reports"
try:
    report_files = sorted(reports_dir.glob("*.txt")) + sorted(reports_dir.glob("*.html"))
    for rf in report_files[-6:]:
        print(f"  {rf.name}  ({rf.stat().st_size:,} bytes)")
    if not report_files:
        print("  No report files found in reports/ directory.")
except Exception as e:
    print(f"  Could not list reports: {e}")

print()
print("Pipeline demo complete.")
